# Financial Advisor Bot – Model Development

This notebook implements the core machine learning pipeline for my Financial Advisor Bot.  
The goals of this notebook are:

1. Download historical ETF data (SPY, QQQ, VEU, AGG).
2. Compute technical indicators (moving averages, MACD, RSI, volatility, momentum, volume signals).
3. Create a 10-day ahead direction label (UP/DOWN).
4. Train a classification model (Random Forest) to predict future direction.
5. Evaluate the model using accuracy and a confusion matrix.
6. Implement simple advisor logic that converts predictions into recommendations and explanations.

This notebook forms the **prototype** for the project and will be used as the basis for the feature prototype chapter and the video demo.

## 1. Imports and Setup

In this section, I import all the Python libraries needed for data handling, technical indicator calculation, machine learning and visualisation.

In [ ]:
pip install pandas numpy matplotlib scikit-learn yfinance seaborn

In [ ]:
# Core libraries
import numpy as np
import pandas as pd

# Data source
import yfinance as yf

# Plotting
import matplotlib.pyplot as plt

# Machine learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# For reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Make plots a bit nicer
plt.style.use("seaborn-v0_8-whitegrid")

print("Imports completed.")

## 2. Download and Inspect ETF Data

In this section, I download 10 years of daily adjusted close prices and trading volumes for a small set of broad-market ETFs:

- SPY – S&P 500 (US equities)
- QQQ – NASDAQ-100 (US technology)
- VEU – FTSE All-World ex-US (international equities)
- AGG – US Aggregate Bond Index (bonds)

These ETFs provide diversified exposure to different parts of the global market and are used as the basis for the advisor's recommendations.

In [ ]:
# List of ETFs to analyse
tickers = ["SPY", "QQQ", "VEU", "AGG"]

start_date = "2014-01-01"
end_date = "2024-01-01"

print(f"Downloading data from {start_date} to {end_date} for: {tickers}")

# Explicitly set auto_adjust=False so we get 'Adj Close' column
raw_data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=False
)

print("Raw data columns:", raw_data.columns)

# We will mostly work with 'Adj Close' and 'Volume'
prices = raw_data["Adj Close"].copy()
volumes = raw_data["Volume"].copy()

print("Data shape (prices):", prices.shape)
print("Data shape (volumes):", volumes.shape)

prices.head()

In [ ]:
# Plot price history for each ETF
plt.figure(figsize=(10, 6))
for ticker in tickers:
    plt.plot(prices.index, prices[ticker], label=ticker)

plt.title("Adjusted Close Prices (2014–2024)")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend()
plt.tight_layout()
plt.show()

## 3. Computing Technical Indicators (SPY Example)

In this section, I transform the raw SPY price and volume data into a set of technical indicators
that will be used as features for the machine learning model.

For each trading day, I compute:

- Simple Moving Averages (SMA): 10, 30, 50, 100 days
- Exponential Moving Averages (EMA): 12, 26 days
- MACD (EMA12 − EMA26) and MACD signal line (9-day EMA of MACD)
- RSI (14-day Relative Strength Index)
- Volatility: 20-day and 50-day rolling standard deviation of returns
- Momentum returns: 5-day, 10-day and 20-day percentage change
- Volume and 5-day Volume Rate of Change (VROC)

These indicators provide interpretable signals about trend, momentum, volatility and trading activity.
They will form the feature set for the direction prediction model.

In [ ]:
# Work with just one ETF first: SPY
ticker = "SPY"

# Create a DataFrame with price and volume for SPY
df = pd.DataFrame({
    "Close": prices[ticker],
    "Volume": volumes[ticker]
})

# 3.1 Moving Averages (SMA)
df["SMA_10"] = df["Close"].rolling(window=10).mean()
df["SMA_30"] = df["Close"].rolling(window=30).mean()
df["SMA_50"] = df["Close"].rolling(window=50).mean()
df["SMA_100"] = df["Close"].rolling(window=100).mean()

# 3.2 Exponential Moving Averages (EMA)
df["EMA_12"] = df["Close"].ewm(span=12, adjust=False).mean()
df["EMA_26"] = df["Close"].ewm(span=26, adjust=False).mean()

# 3.3 MACD and Signal Line 
df["MACD"] = df["EMA_12"] - df["EMA_26"]
df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

# --- 3.4 RSI (Relative Strength Index, 14-period) ---
delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

window_rsi = 14
avg_gain = gain.rolling(window=window_rsi).mean()
avg_loss = loss.rolling(window=window_rsi).mean()

rs = avg_gain / avg_loss
df["RSI_14"] = 100 - (100 / (1 + rs))

# --- 3.5 Volatility (rolling standard deviation of returns) ---
returns = df["Close"].pct_change()
df["Volatility_20"] = returns.rolling(window=20).std()
df["Volatility_50"] = returns.rolling(window=50).std()

# --- 3.6 Momentum Returns ---
df["Mom_5"] = df["Close"].pct_change(periods=5)
df["Mom_10"] = df["Close"].pct_change(periods=10)
df["Mom_20"] = df["Close"].pct_change(periods=20)

# --- 3.7 Volume-based Feature: 5-day Volume Rate of Change ---
df["VROC_5"] = df["Volume"].pct_change(periods=5)

# Drop rows with NaN created by rolling calculations
df_indicators = df.dropna().copy()

print("Indicator DataFrame shape:", df_indicators.shape)
df_indicators.head()

Creates a df with Close price + Volume for SPY
	•	Adds:
	•	SMA (smooth trend over different window sizes)
	•	EMA (more weight on recent prices)
	•	MACD & signal (momentum indicator)
	•	RSI (overbought/oversold indicator)
	•	Volatility (how “bouncy” price is)
	•	Momentum (recent percentage change)
	•	VROC (how volume has changed over 5 days)
	•	Drops the first rows where rolling windows don’t have enough data
	•	Prints the size and shows the first few rows

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_indicators.index, df_indicators["Close"], label="SPY Close")
plt.plot(df_indicators.index, df_indicators["SMA_50"], label="SMA 50")
plt.plot(df_indicators.index, df_indicators["SMA_100"], label="SMA 100")
plt.title("SPY Close vs 50/100-day SMA")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend()
plt.tight_layout()
plt.show()

## 4. Creating the 10-Day Direction Label

The machine learning model needs a target variable that indicates whether the ETF's price
moves upward or downward over a future window. Here, I create a **10-day ahead direction label**:

- If the price after 10 trading days is higher than today's price → label = 1 (UP)
- If the price after 10 days is lower → label = 0 (DOWN)

This transforms the problem into a **binary classification task**, which aligns with the project
goal of predicting short-term market direction rather than exact price values.

After generating the label, I align it with the technical indicator dataset and remove rows where
future data is unavailable (e.g. the last 10 rows).

In [ ]:
# Number of days ahead to predict
FUTURE_HORIZON = 10

# Create a column for future price
df_indicators["Future_Close"] = df_indicators["Close"].shift(-FUTURE_HORIZON)

# Create direction label: 1 = Up, 0 = Down
df_indicators["Target"] = (df_indicators["Future_Close"] > df_indicators["Close"]).astype(int)

# Drop last FUTURE_HORIZON rows because they have no future data
df_model = df_indicators.dropna().copy()

print("Final modelling DataFrame shape:", df_model.shape)
df_model[["Close", "Future_Close", "Target"]].head(10)

What this does:
- Takes today’s price
- Shifts the price 10 days into the future
- Compares future vs current

Sets:
- Target = 1 if future price is higher
- Target = 0 if future price is lower
- Drops rows with missing values from the end

This gives us thousands of training examples from 10 years of data

## 5. Machine Learning Model: Training and Evaluation

In this section, I prepare the dataset for machine learning and train a Random Forest classifier
to predict whether SPY will move up or down over the next 10 trading days.

Steps:
1. Select technical indicator columns as features.
2. Use a chronological train–test split (train on older data, test on recent data).
3. Train the Random Forest classifier.
4. Evaluate its performance using direction accuracy and a confusion matrix.

A chronological split is required because shuffling financial data would lead to information leakage.
This structure mimics real-world forecasting where models learn from the past and are evaluated
on future unseen data.

In [ ]:
# 5.1 Select feature columns
feature_cols = [
    "SMA_10", "SMA_30", "SMA_50", "SMA_100",
    "EMA_12", "EMA_26",
    "MACD", "MACD_signal",
    "RSI_14",
    "Volatility_20", "Volatility_50",
    "Mom_5", "Mom_10", "Mom_20",
    "VROC_5",
    "Volume"
]

X = df_model[feature_cols].copy()
y = df_model["Target"].copy()

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

# 5.2 Chronological train-test split

split_index = int(len(X) * 0.8)   # 80% train, 20% test

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
# 5.3 Train Random Forest

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=RANDOM_SEED
)

model.fit(X_train, y_train)

print("Model training complete.")

## 6. Model Evaluation

After training the model, I evaluate its performance using:

- Direction accuracy (percentage of UP/DOWN predictions correctly classified)
- Confusion matrix (how many UPs and DOWNs were correctly or incorrectly predicted)
- Feature importance ranking (which indicators the model relied on most)

These metrics help determine how well the model captures short-term market behaviour.

In [ ]:
# 6.1 Predictions 
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Direction Accuracy: {accuracy * 100:.2f}%")

# 6.2 Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - SPY 10-Day Direction Prediction")
plt.show()

In [ ]:
# 6.3 Feature Importance
importances = model.feature_importances_
sorted_idx = np.argsort(importances)

plt.figure(figsize=(8, 6))
plt.barh(np.array(feature_cols)[sorted_idx], importances[sorted_idx], color="teal")
plt.title("Feature Importance – Random Forest")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

## 7. Advisor Logic and Explanation Engine

Up to this point, I have trained a model that can predict whether SPY is likely to move up or down
over the next 10 trading days. However, a financial advisor bot must do more than output a label:
it must translate the prediction into **actionable guidance** and provide a **human-readable explanation**.

In this section, I implement a simple rule-based advisor layer on top of the model:

- If the model predicts an upward move and key trend/momentum indicators are supportive,
  the advisor suggests increasing or maintaining exposure.
- If the model predicts a downward move or volatility and negative momentum are high,
  the advisor suggests reducing exposure or exercising caution.

The explanation engine uses technical indicators such as SMA crossovers, MACD, RSI, volatility and
momentum to generate a short textual justification. This demonstrates how the system can provide
transparent and interpretable recommendations to a non-technical user.

In [ ]:
# Create a copy of the test portion of the modelling DataFrame
df_test = df_model.iloc[split_index:].copy()

# Add model predictions
df_test["Predicted"] = y_pred
df_test["Actual"] = y_test.values

# Inspect last few rows (most recent data points)
df_test.tail()

In [ ]:
def generate_advice_and_explanation(row, investment_amount=1000):
    """
    Generate a simple recommendation and explanation for a single day
    based on the model prediction and technical indicators.
    """
    prediction = row["Predicted"]
    close_price = row["Close"]
    
    reasons = []
    
    # Trend-based reasoning using SMAs
    if row["SMA_30"] > row["SMA_100"]:
        reasons.append("short-term trend is stronger than the long-term trend (SMA 30 > SMA 100)")
    else:
        reasons.append("short-term trend is weaker than or similar to the long-term trend (SMA 30 ≤ SMA 100)")
    
    # MACD momentum signal
    if row["MACD"] > 0:
        reasons.append("MACD is positive, indicating bullish momentum")
    else:
        reasons.append("MACD is negative or weak, indicating limited or bearish momentum")
    
    # RSI – not overbought / overbought
    if row["RSI_14"] < 30:
        reasons.append("RSI is below 30, suggesting the asset may be oversold")
    elif row["RSI_14"] > 70:
        reasons.append("RSI is above 70, suggesting the asset may be overbought")
    else:
        reasons.append("RSI is in a neutral zone (between 30 and 70)")
    
    # Volatility and momentum context
    if row["Volatility_20"] > row["Volatility_50"]:
        reasons.append("short-term volatility is higher than usual")
    
    if row["Mom_10"] > 0:
        reasons.append("recent 10-day momentum has been positive")
    else:
        reasons.append("recent 10-day momentum has been negative or flat")
    
    # Build explanation text
    explanation = " ; ".join(reasons)
    
    # Simple allocation logic based on prediction and indicators
    if prediction == 1:
        # Bullish
        recommendation = (
            f"The model predicts that SPY is likely to rise over the next 10 trading days. "
            f"For an investment amount of ${investment_amount:.0f}, a **moderate to higher allocation** to SPY "
            f"could be considered, depending on your overall risk tolerance."
        )
    else:
        # Bearish / cautious
        recommendation = (
            f"The model predicts that SPY is unlikely to rise over the next 10 trading days. "
            f"For an investment amount of ${investment_amount:.0f}, a **reduced allocation** or a more defensive "
            f"position in SPY may be appropriate, especially if you are risk-averse."
        )
    
    return recommendation, explanation

In [ ]:
# Pick the last 5 test rows (most recent dates)
sample_rows = df_test.tail(5)

for idx, row in sample_rows.iterrows():
    rec, expl = generate_advice_and_explanation(row, investment_amount=5000)
    
    print("=" * 80)
    print(f"Date: {idx.date()} | Close price: ${row['Close']:.2f}")
    print(f"Model Prediction (10-day direction): {'UP' if row['Predicted'] == 1 else 'DOWN'}")
    print("\nRecommendation:")
    print(rec)
    print("\nExplanation:")
    print(expl)
    print()

In [ ]:
print("Training Start Date:", X_train.index.min())
print("Training End Date:", X_train.index.max())

print("\nTesting Start Date:", X_test.index.min())
print("Testing End Date:", X_test.index.max())

## 8. Simple User Interface (Prototype)

In this section, I implement a minimal user interface inside the notebook so a non-technical user
can interact with the advisor prototype.

The UI allows the user to:
- choose an investment amount and risk level,
- select a date from the testing period,
- receive a prediction (UP/DOWN), a recommendation, and an explanation.

This prototype UI demonstrates how the system can later be transferred into a web application
(Flask/Streamlit) with the same core logic.

In [ ]:
# Section 8: Simple UI inside Jupyter (ipywidgets)
import ipywidgets as widgets
from IPython.display import display, clear_output

# Safety check: ensure df_test exists 
# df_test should have: Close, indicators, Predicted, Actual
required_cols = {"Close", "Predicted", "Actual", "SMA_30", "SMA_100", "MACD", "RSI_14", "Volatility_20", "Volatility_50", "Mom_10"}
missing = required_cols - set(df_test.columns)
if missing:
    print("Missing columns in df_test:", missing)
    print("Make sure you have run Section 7 (df_test + predictions) before running Section 8.")
else:
    # Widgets
    date_dropdown = widgets.Dropdown(
        options=[(str(d.date()), d) for d in df_test.index],
        description="Date:",
        layout=widgets.Layout(width="380px")
    )

    amount_slider = widgets.IntSlider(
        value=5000, min=100, max=50000, step=100,
        description="Amount ($):",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="500px")
    )

    risk_dropdown = widgets.Dropdown(
        options=["Low", "Moderate", "High"],
        value="Moderate",
        description="Risk Level:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="250px")
    )

    run_button = widgets.Button(
        description="Get Advice",
        button_style="success",
        icon="check"
    )

    output = widgets.Output()

    # Simple allocation suggestion based on risk + prediction 
    def allocation_hint(predicted_up, risk_level):
        # These are intentionally simple (prototype-level) and explainable
        if risk_level == "Low":
            return "Low-risk: keep allocation smaller and prioritise stability."
        if risk_level == "High":
            return "High-risk: you may accept larger exposure if signals are positive."
        return "Moderate-risk: balanced exposure depending on signals."

    # Action handler
    def on_run_clicked(_):
        with output:
            clear_output()
            chosen_date = date_dropdown.value
            row = df_test.loc[chosen_date]

            rec, expl = generate_advice_and_explanation(row, investment_amount=amount_slider.value)

            pred_text = "UP" if row["Predicted"] == 1 else "DOWN"
            actual_text = "UP" if row["Actual"] == 1 else "DOWN"

            print("=== Financial Advisor Bot (Prototype UI) ===")
            print(f"Asset: SPY")
            print(f"Date selected: {chosen_date.date()}")
            print(f"Close price: ${row['Close']:.2f}")
            print(f"Prediction (next 10 trading days): {pred_text}")
            print(f"Actual outcome (for evaluation): {actual_text}")
            print()
            print("Recommendation:")
            print(rec)
            print()
            print("Explanation (indicator-based):")
            print(expl)
            print()
            print("Risk note:")
            print(allocation_hint(row["Predicted"] == 1, risk_dropdown.value))

    run_button.on_click(on_run_clicked)

    # --- Display UI ---
    ui_row1 = widgets.HBox([date_dropdown, risk_dropdown])
    ui_row2 = widgets.HBox([amount_slider, run_button])

    display(ui_row1, ui_row2, output)

    print("UI loaded. Select a date + risk + amount, then click 'Get Advice'.")

In [ ]:
import joblib
import os

# Create models folder if it doesn't exist
os.makedirs("models", exist_ok=True)

# Save the trained Random Forest model
joblib.dump(model, "models/rf_model_spy.pkl")

print("Model saved successfully.")

In [ ]:
# Create data folder if it doesn't exist
os.makedirs("data", exist_ok=True)

# Save processed SPY dataset (with indicators)
df_model.to_csv("data/spy_processed.csv")

print("Processed data saved successfully.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Step 1: Put predictions into df_test ---
df_test = df_test.copy()
df_test['Actual'] = y_test
df_test['Prediction'] = y_pred

# --- Step 2: Calculate daily returns ---
df_test['Market_Return'] = df_test['Close'].pct_change()

# --- Step 3: Shift prediction (VERY IMPORTANT) ---
df_test['Strategy_Return'] = df_test['Market_Return'] * df_test['Prediction'].shift(1)

# --- Step 4: Replace NaN values ---
df_test = df_test.dropna()

# --- Step 5: Compute cumulative returns ---
df_test['Market_Cumulative'] = (1 + df_test['Market_Return']).cumprod()
df_test['Strategy_Cumulative'] = (1 + df_test['Strategy_Return']).cumprod()

# --- Step 6: Plot ---
plt.figure(figsize=(12,6))

plt.plot(df_test.index, df_test['Market_Cumulative'], label='Buy & Hold', linewidth=2)
plt.plot(df_test.index, df_test['Strategy_Cumulative'], label='ML Strategy', linewidth=2)

plt.title('Equity Curve: ML Strategy vs Buy & Hold')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.grid()

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Down (0)', 'Up (1)'],
            yticklabels=['Down (0)', 'Up (1)'])

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Get feature importance
importance = model.feature_importances_
features = X.columns

# Create DataFrame
feat_imp = pd.DataFrame({
    'Feature': features,
    'Importance': importance
}).sort_values(by='Importance', ascending=False)

# Plot top 10 features
plt.figure(figsize=(10,6))
plt.barh(feat_imp['Feature'][:10], feat_imp['Importance'][:10])
plt.gca().invert_yaxis()

plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance Score')
plt.ylabel('Features')

plt.show()